# Retrain clean-input GRU and CNN velocity models

This notebook is the reproducible bridge between the IDR deterministic sensor pipeline and the teammate's speed-model data. It retrains two new models from scratch on cleaned vehicle-frame IMU windows: a six-channel GRU and a comparable six-channel temporal CNN.

It deliberately does not load the existing GRU or CNN checkpoints. Those checkpoints were trained on raw phone-frame channels, include gravity, use km/h targets, and do not contain the scaler artifacts needed for safe deployment.

Important dataset note: the CAN-bus indicated vehicle speed remains the supervised label. GPS speed from the phone is used only during the prefix-only calibration proxy; it is never a model feature or label.

## What the notebook does

For each recording it: (1) reads the paired phone and vehicle CSV files using their CP-1252 encoding, (2) aligns the target by relative time, (3) derives rolling, past-only mounting evidence from phone accelerometer, gyro, and GPS-speed changes and feeds it to DynamicVehicleCalibrator, (4) sends every row through the IDR normalisation, orientation, calibration, gravity-removal, quality, resampling, and windowing functions, then (5) trains and compares a GRU and CNN using journey-disjoint train/validation/test splits.

The mounting-evidence heuristic is an offline training approximation, not the final live calibration policy. It is nevertheless causal: every evidence window contains only readings observed at or before its timestamp, and the model receives no window until DynamicVehicleCalibrator has published a sufficiently confident calibration.

In [1]:
# Run from the repository root or from notebooks/. No package install is
# required for the local source tree: src/ is added explicitly below.
from __future__ import annotations

import json
import random
import sys
import time
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'idr_backend').is_dir():
            return candidate
    raise RuntimeError('Open the notebook inside the dead reckoning repository.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

RAW_DATA_DIR = REPO_ROOT / 'SIH-2-main' / 'SIH-2-main' / 'IOVNBD-Speed-Prediction' / 'data' / 'raw'
ARTIFACT_DIR = REPO_ROOT / 'artifacts' / 'clean_velocity_comparison'
assert RAW_DATA_DIR.is_dir(), f'Missing raw data directory: {RAW_DATA_DIR}'

print(f'Repository: {REPO_ROOT}')
print(f'Raw data:   {RAW_DATA_DIR}')
print(f'Artifacts:  {ARTIFACT_DIR}')

Repository: E:\dead reckoning
Raw data:   E:\dead reckoning\SIH-2-main\SIH-2-main\IOVNBD-Speed-Prediction\data\raw
Artifacts:  E:\dead reckoning\artifacts\clean_velocity_comparison


In [3]:
# Dependencies are intentionally checked, not installed by the notebook.
# Use a Python 3.12 environment for the declared project support. The
# available local CUDA environment may be newer, but is not the support contract.
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# These are the production deterministic functions, not notebook duplicates.
from idr_backend.sensors.calibration import CalibrationEvidence, DynamicVehicleCalibrator, derive_sensor_to_vehicle_quaternion, rotate_imu_to_vehicle
from idr_backend.sensors.gravity_removal import remove_gravity_from_vehicle_imu
from idr_backend.sensors.normalization import normalize_raw_sample
from idr_backend.sensors.orientation import ImuOrientationEstimator, normalize_vector
from idr_backend.sensors.quality import VehicleImuQualityLimits, VehicleImuQualityMonitor
from idr_backend.sensors.resampling import FixedRateVehicleImuResampler
from idr_backend.sensors.synchronization import synchronize_pair
from idr_backend.sensors.types import CoordinateFrame, MeasurementUnit, RawSensorSample, SensorKind, SensorSource, VehicleCalibration
from idr_backend.sensors.windowing import CausalVehicleImuWindowBuilder, VELOCITY_MODEL_FEATURE_NAMES, velocity_model_feature_rows

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}; device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'torch'

In [ ]:
@dataclass(frozen=True)
class RetrainConfig:
    # Model shape: 5 s windows at the measured 10 Hz data rate.
    sample_period_ns: int = 100_000_000
    window_size: int = 50
    window_stride: int = 5

    # Rolling, past-only mounting-evidence window used by the offline proxy.
    calibration_history_s: float = 30.0
    calibration_update_every_samples: int = 10
    # Five points are required after each point has been aggregated over a
    # 30-second trailing window; this is an IO-VNBD dataset setting, not a
    # production recommendation.
    minimum_calibration_samples: int = 5
    label_alignment_tolerance_ms: float = 75.0

    # Deterministic quality gates. Tune from held-out data before deployment.
    max_gap_ns: int = 250_000_000
    max_linear_acceleration_mps2: float = 30.0
    max_angular_velocity_radps: float = 12.0
    minimum_calibration_confidence: float = 0.60

    # Both models use exactly the same split, scaled features, and labels.
    seed: int = 42
    train_fraction: float = 0.70
    validation_fraction: float = 0.15
    test_fraction: float = 0.15
    batch_size: int = 256
    max_epochs: int = 50
    early_stopping_patience: int = 8
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4

    # Selection uses only non-test journeys. The final test journeys stay
    # untouched until exactly one final model per family is fitted.
    cv_folds: int = 3
    tuning_max_epochs: int = 25
    tuning_early_stopping_patience: int = 5
    final_max_epochs: int = 50
    inference_warmup_calls: int = 30
    inference_measurement_calls: int = 200

CONFIG = RetrainConfig()
assert CONFIG.window_size * CONFIG.sample_period_ns == 5_000_000_000
assert np.isclose(CONFIG.train_fraction + CONFIG.validation_fraction + CONFIG.test_fraction, 1.0)
print(asdict(CONFIG))

## Read and align the paired recordings

The source CSVs use CP-1252, not UTF-8. Some files have unequal S/V row counts. The alignment below uses the relative time in each recording and rejects a label when no vehicle reading lies within 75 ms. This is deliberately safer than row truncation.

In [ ]:
def find_column(columns, *required_fragments: str) -> str:
    matches = [column for column in columns if all(fragment.lower() in column.lower() for fragment in required_fragments)]
    if len(matches) != 1:
        raise KeyError(f'Expected one column containing {required_fragments}; found {matches}')
    return matches[0]

def paired_paths(raw_dir: Path) -> list[tuple[str, Path, Path]]:
    pairs = []
    for sensor_path in sorted(raw_dir.glob('S-*.csv')):
        vehicle_path = raw_dir / f'V-{sensor_path.name[2:]}'
        if vehicle_path.exists():
            pairs.append((sensor_path.stem[2:], sensor_path, vehicle_path))
    if not pairs:
        raise RuntimeError(f'No S-/V- pairs found under {raw_dir}')
    return pairs

def read_aligned_sequence(sequence_id: str, sensor_path: Path, vehicle_path: Path, config: RetrainConfig) -> pd.DataFrame:
    sensor = pd.read_csv(sensor_path, encoding='cp1252')
    vehicle = pd.read_csv(vehicle_path, encoding='cp1252')

    date = find_column(sensor.columns, 'DATE')
    acc_x, acc_y, acc_z = (find_column(sensor.columns, 'ACCELEROMETER', axis) for axis in ('X', 'Y', 'Z'))
    gyro_x = find_column(sensor.columns, 'GYROSCOPE', 'Roll')
    gyro_y = find_column(sensor.columns, 'GYROSCOPE', 'Pitch')
    gyro_z = find_column(sensor.columns, 'GYROSCOPE', 'Yaw')
    gps_speed = find_column(sensor.columns, 'GPS SPEED')
    target_time = find_column(vehicle.columns, 'Time Since Start of Day')
    target_speed = find_column(vehicle.columns, 'Indicated Vehicle Speed')

    sensor_time = pd.to_datetime(sensor[date], format='%Y-%m-%d %H:%M:%S:%f', errors='coerce')
    s = pd.DataFrame({
        'timestamp': sensor_time,
        'acc_x': pd.to_numeric(sensor[acc_x], errors='coerce'),
        'acc_y': pd.to_numeric(sensor[acc_y], errors='coerce'),
        'acc_z': pd.to_numeric(sensor[acc_z], errors='coerce'),
        'gyro_x': pd.to_numeric(sensor[gyro_x], errors='coerce'),
        'gyro_y': pd.to_numeric(sensor[gyro_y], errors='coerce'),
        'gyro_z': pd.to_numeric(sensor[gyro_z], errors='coerce'),
        'gps_speed_kmh': pd.to_numeric(sensor[gps_speed], errors='coerce'),
    }).dropna(subset=['timestamp']).sort_values('timestamp').drop_duplicates('timestamp')
    s['relative_ns'] = (s['timestamp'] - s['timestamp'].iloc[0]).dt.total_seconds().mul(1e9).round().astype('int64')

    v = pd.DataFrame({
        'vehicle_relative_s': pd.to_numeric(vehicle[target_time], errors='coerce'),
        'target_speed_kmh': pd.to_numeric(vehicle[target_speed], errors='coerce'),
    }).dropna().sort_values('vehicle_relative_s')
    v['relative_ns'] = (v['vehicle_relative_s'] - v['vehicle_relative_s'].iloc[0]).mul(1e9).round().astype('int64')
    v = v.drop_duplicates('relative_ns')[['relative_ns', 'target_speed_kmh']]

    aligned = pd.merge_asof(s.sort_values('relative_ns'), v, on='relative_ns', direction='nearest', tolerance=int(config.label_alignment_tolerance_ms * 1e6))
    aligned = aligned.dropna(subset=['target_speed_kmh']).copy()
    numeric = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'target_speed_kmh']
    aligned = aligned.replace([np.inf, -np.inf], np.nan).dropna(subset=numeric)
    aligned['timestamp_ns'] = aligned['timestamp'].astype('int64')
    aligned['target_speed_mps'] = aligned['target_speed_kmh'] / 3.6
    aligned['sequence_id'] = sequence_id
    return aligned.reset_index(drop=True)

pairs = paired_paths(RAW_DATA_DIR)
print(f'Found {len(pairs)} paired recordings.')
example = read_aligned_sequence(*pairs[0], CONFIG)
example[['sequence_id', 'timestamp', 'target_speed_kmh']].head()

## Causal rolling calibration evidence

The calibration function must be given vehicle up and a signed vehicle-forward direction in sensor coordinates. This helper estimates up from low-dynamics specific-force samples in a trailing 30-second window. It estimates forward by correlating gravity-compensated sensor acceleration with the *phone GPS-speed derivative*. It does not look at the CAN-bus label.

This is a transparent, conservative approximation for producing training features. If the evidence is weak or axes are nearly parallel, it emits no calibration evidence rather than guessing. DynamicVehicleCalibrator decides when coherent evidence is sufficient.

In [ ]:
GRAVITY_MPS2 = 9.80665

def rolling_calibration_evidence(history: deque[tuple[float, ...]], source_id: str, config: RetrainConfig) -> CalibrationEvidence | None:
    if len(history) < config.minimum_calibration_samples:
        return None
    # Keep session nanoseconds as int64. Converting epoch timestamps to a
    # floating array loses precision and can make evidence look newer than
    # the sample that produced it.
    timestamps_ns = np.fromiter((int(item[0]) for item in history), dtype=np.int64)
    values = np.asarray([item[1:] for item in history], dtype=float)
    acceleration, gyro, gps_speed_kmh = values[:, :3], values[:, 3:6], values[:, 6]
    acceleration_norm = np.linalg.norm(acceleration, axis=1)
    gyro_norm = np.linalg.norm(gyro, axis=1)
    quiet = (np.abs(acceleration_norm - GRAVITY_MPS2) <= 2.0) & (gyro_norm <= 1.5)
    if int(quiet.sum()) < config.minimum_calibration_samples:
        return None
    vehicle_up_in_sensor = normalize_vector(tuple(np.median(acceleration[quiet], axis=0)))
    time_s = (timestamps_ns - timestamps_ns[0]) * 1e-9
    speed_mps = gps_speed_kmh / 3.6
    longitudinal_acceleration = np.gradient(speed_mps, time_s)
    linear_sensor = acceleration - GRAVITY_MPS2 * np.asarray(vehicle_up_in_sensor)
    useful = np.isfinite(longitudinal_acceleration) & (np.abs(longitudinal_acceleration) >= 0.20) & (np.abs(longitudinal_acceleration) <= 6.0) & (gyro_norm <= 1.5)
    if int(useful.sum()) < config.minimum_calibration_samples:
        return None
    try:
        vehicle_forward_in_sensor = normalize_vector(tuple((linear_sensor[useful] * longitudinal_acceleration[useful, None]).sum(axis=0)))
        # Validate that the two inferred axes are non-parallel before publishing.
        derive_sensor_to_vehicle_quaternion(vehicle_up_in_sensor, vehicle_forward_in_sensor)
    except ValueError:
        return None
    confidence = min(1.0, min(int(quiet.sum()), int(useful.sum())) / (2 * config.minimum_calibration_samples))
    return CalibrationEvidence(int(timestamps_ns[-1]), SensorSource.PHONE, source_id, vehicle_up_in_sensor, vehicle_forward_in_sensor, confidence)

In [ ]:
def nearest_target(frame: pd.DataFrame, timestamp_ns: int, tolerance_ns: int) -> float | None:
    times = frame['timestamp_ns'].to_numpy(np.int64)
    values = frame['target_speed_mps'].to_numpy(float)
    index = int(np.searchsorted(times, timestamp_ns))
    options = [candidate for candidate in (index - 1, index) if 0 <= candidate < len(times)]
    if not options:
        return None
    closest = min(options, key=lambda candidate: abs(int(times[candidate]) - timestamp_ns))
    return float(values[closest]) if abs(int(times[closest]) - timestamp_ns) <= tolerance_ns else None

def replay_clean_sequence(frame: pd.DataFrame, config: RetrainConfig) -> tuple[np.ndarray, np.ndarray, dict[str, int | float | str]]:
    source_id = f'dataset-phone-{frame.sequence_id.iloc[0]}'
    orientation = ImuOrientationEstimator(accelerometer_correction_gain_per_s=0.5, acceleration_trust_tolerance_mps2=1.5)
    quality_monitor = VehicleImuQualityMonitor(VehicleImuQualityLimits(config.max_gap_ns, config.max_linear_acceleration_mps2, config.max_angular_velocity_radps, config.minimum_calibration_confidence))
    resampler = FixedRateVehicleImuResampler(config.sample_period_ns)
    window_builder = CausalVehicleImuWindowBuilder(window_size=config.window_size, sample_period_ns=config.sample_period_ns)
    calibrator = DynamicVehicleCalibrator(minimum_evidence_count=3, minimum_evidence_confidence=config.minimum_calibration_confidence, maximum_disagreement_rad=0.70, remount_evidence_count=3)
    history: deque[tuple[float, ...]] = deque(maxlen=int(config.calibration_history_s * 1e9 / config.sample_period_ns))
    rows, targets = [], []
    accepted, rejected = 0, 0
    label_tolerance_ns = int(config.label_alignment_tolerance_ms * 1e6)

    for sample_number, row in enumerate(frame.itertuples(index=False), start=1):
        raw_acceleration = RawSensorSample(int(row.timestamp_ns), SensorSource.PHONE, source_id, SensorKind.ACCELEROMETER, (float(row.acc_x), float(row.acc_y), float(row.acc_z)), MeasurementUnit.METERS_PER_SECOND_SQUARED, CoordinateFrame.SENSOR)
        raw_gyro = RawSensorSample(int(row.timestamp_ns), SensorSource.PHONE, source_id, SensorKind.GYROSCOPE, (float(row.gyro_x), float(row.gyro_y), float(row.gyro_z)), MeasurementUnit.RADIANS_PER_SECOND, CoordinateFrame.SENSOR)
        synchronized = synchronize_pair(normalize_raw_sample(raw_acceleration), normalize_raw_sample(raw_gyro), max_skew_ns=0)
        orientation_estimate = orientation.update(synchronized)
        history.append((int(row.timestamp_ns), float(row.acc_x), float(row.acc_y), float(row.acc_z), float(row.gyro_x), float(row.gyro_y), float(row.gyro_z), float(row.gps_speed_kmh)))
        if sample_number % config.calibration_update_every_samples == 0:
            evidence = rolling_calibration_evidence(history, source_id, config)
            if evidence is not None:
                calibrator.update(evidence)
        calibration = calibrator.calibration
        if calibration is None or calibration.confidence < config.minimum_calibration_confidence:
            continue
        vehicle_sample = rotate_imu_to_vehicle(synchronized, calibration, config.minimum_calibration_confidence)
        clean_sample = remove_gravity_from_vehicle_imu(vehicle_sample, orientation_estimate, calibration)
        quality = quality_monitor.assess(clean_sample)
        if not quality.is_acceptable:
            rejected += 1
        for resampled in resampler.push(clean_sample, quality):
            window = window_builder.push(resampled)
            if window is None:
                continue
            target = nearest_target(frame, window.end_timestamp_ns, label_tolerance_ns)
            if target is None:
                continue
            accepted += 1
            if accepted % config.window_stride == 0:
                rows.append(velocity_model_feature_rows(window))
                targets.append(target)

    if not rows:
        raise ValueError('This sequence produced no complete quality-gated windows.')
    audit = {'sequence_id': str(frame.sequence_id.iloc[0]), 'source_rows': len(frame), 'windows': len(rows), 'quality_rejections': rejected, 'calibration_confidence': calibration.confidence if calibration else 0.0, 'calibration_phase': calibrator.phase}
    return np.asarray(rows, dtype=np.float32), np.asarray(targets, dtype=np.float32), audit

## Build a leakage-safe dataset

Run this cell once. It may take several minutes because it intentionally calls the deterministic Python pipeline once per raw sample. It prints skipped sequences and audit facts rather than manufacturing data when a recording never produces sufficiently coherent past-only calibration evidence.

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG.seed)
all_windows, all_targets, all_groups, audits, skipped = [], [], [], [], []
for sequence_id, sensor_path, vehicle_path in pairs:
    try:
        frame = read_aligned_sequence(sequence_id, sensor_path, vehicle_path, CONFIG)
        windows, targets, audit = replay_clean_sequence(frame, CONFIG)
        all_windows.append(windows)
        all_targets.append(targets)
        all_groups.extend([sequence_id] * len(targets))
        audits.append(audit)
        print(f'{sequence_id}: {audit}')
    except (KeyError, ValueError) as error:
        skipped.append({'sequence_id': sequence_id, 'reason': str(error)})
        print(f'{sequence_id}: skipped - {error}')

if len(all_windows) < 3:
    raise RuntimeError('Too few calibrated journeys to create group-disjoint splits.')
X = np.concatenate(all_windows)
y_mps = np.concatenate(all_targets)
groups = np.asarray(all_groups)
audit_frame = pd.DataFrame(audits)
skipped_frame = pd.DataFrame(skipped)
print(f'Final clean windows: {X.shape}; target unit: m/s; journeys: {len(set(groups))}')
audit_frame

In [ ]:
# Split by journey, never by overlapping windows. A random group split can
# put every long journey in train, leaving a meaningless test set. This
# greedy allocator retains whole journeys while aiming for window-count
# fractions of 70/15/15. Fit scalers on train journeys only afterwards.
def balanced_group_indices(groups: np.ndarray, config: RetrainConfig) -> dict[str, np.ndarray]:
    group_counts = pd.Series(groups.astype(str)).value_counts().to_dict()
    if len(group_counts) < 3:
        raise ValueError('At least three journeys are required for train/validation/test splitting.')
    targets = {'train': len(groups) * config.train_fraction, 'validation': len(groups) * config.validation_fraction, 'test': len(groups) * config.test_fraction}
    totals = {name: 0 for name in targets}
    assigned = {name: [] for name in targets}
    tie_breaker = {name: value for value, name in enumerate(np.random.default_rng(config.seed).permutation(list(group_counts)))}
    ranked_sequences = sorted(group_counts, key=lambda name: (-group_counts[name], tie_breaker[name]))
    # The source journeys are extremely uneven. Seed the three largest into
    # different splits before balancing the rest, otherwise train may contain
    # only the two dominant journeys and teach us little about generalisation.
    for split, sequence_id in zip(('train', 'validation', 'test'), ranked_sequences[:3]):
        totals[split] += group_counts[sequence_id]
        assigned[split].append(sequence_id)
    for sequence_id in ranked_sequences[3:]:
        count = group_counts[sequence_id]
        def projected_error(split: str) -> float:
            return sum(((totals[name] + (count if name == split else 0)) - targets[name]) ** 2 for name in targets)
        chosen = min(targets, key=projected_error)
        totals[chosen] += count
        assigned[chosen].append(sequence_id)
    assert all(assigned.values()), f'An empty split would be unsafe: {assigned}'
    return {name: np.flatnonzero(np.isin(groups.astype(str), sequence_ids)) for name, sequence_ids in assigned.items()}

indices = balanced_group_indices(groups, CONFIG)
train_index, validation_index, test_index = indices['train'], indices['validation'], indices['test']

feature_scaler = StandardScaler().fit(X[train_index].reshape(-1, X.shape[-1]))
target_scaler = StandardScaler().fit(y_mps[train_index].reshape(-1, 1))
X_scaled = feature_scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape).astype(np.float32)
y_scaled = target_scaler.transform(y_mps.reshape(-1, 1)).astype(np.float32).ravel()

def loader(indices: np.ndarray, shuffle: bool) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(X_scaled[indices]), torch.from_numpy(y_scaled[indices]))
    return DataLoader(dataset, batch_size=CONFIG.batch_size, shuffle=shuffle, num_workers=0, pin_memory=DEVICE.type == 'cuda')

train_loader, validation_loader, test_loader = loader(train_index, True), loader(validation_index, False), loader(test_index, False)
split_summary = {name: {'windows': len(index), 'journeys': sorted(map(str, set(groups[index])))} for name, index in {'train': train_index, 'validation': validation_index, 'test': test_index}.items()}
print(split_summary)
if len(split_summary['validation']['journeys']) < 3:
    print('WARNING: journey lengths are highly imbalanced, so validation contains fewer than three journeys. The CV stage below uses all non-test journeys for selection instead.')

# Test groups are frozen now. The prior train/validation allocation only
# establishes that final holdout; both are combined for CV development.
development_index = np.sort(np.concatenate([train_index, validation_index]))

def balanced_group_folds(indices: np.ndarray, groups: np.ndarray, folds: int, seed: int) -> list[tuple[np.ndarray, np.ndarray]]:
    counts = pd.Series(groups[indices].astype(str)).value_counts().to_dict()
    if len(counts) < folds:
        raise ValueError(f'Need at least {folds} development journeys for group CV.')
    tie_breaker = {name: rank for rank, name in enumerate(np.random.default_rng(seed).permutation(list(counts)))}
    fold_groups = [[] for _ in range(folds)]
    fold_windows = [0] * folds
    for sequence_id in sorted(counts, key=lambda name: (-counts[name], tie_breaker[name])):
        fold = min(range(folds), key=lambda number: fold_windows[number])
        fold_groups[fold].append(sequence_id)
        fold_windows[fold] += counts[sequence_id]
    result = []
    development_groups = groups[development_index].astype(str)
    for held_groups in fold_groups:
        held_mask = np.isin(development_groups, held_groups)
        result.append((development_index[~held_mask], development_index[held_mask]))
    return result

cv_folds = balanced_group_folds(development_index, groups, CONFIG.cv_folds, CONFIG.seed)
print({'development_windows': len(development_index), 'development_journeys': len(set(groups[development_index])), 'frozen_test_windows': len(test_index), 'frozen_test_journeys': len(set(groups[test_index])), 'cv_validation_windows': [len(validation) for _, validation in cv_folds]})

## Selection protocol and comparable models

The final test journeys are frozen before tuning. Every candidate is selected by three group-aware folds over the development journeys, scored by macro journey MAE: each journey contributes equally even if it has many more windows. The test score is calculated once only after choosing a candidate and refitting it.

Bounded search grid: GRU has 3 candidates (32 or 64 hidden units, 1 or 2 layers, dropout 0.10 or 0.20, learning rate 0.0005 or 0.001); CNN has 3 channel/dropout/learning-rate candidates; Random Forest has 4 structural candidates screened with 100 trees, then the selected structure is refit with 300 trees. This is a deliberate small-data search, not an unbounded expensive sweep.

Both neural models receive [batch, 50, 6] standardized clean features and predict standardized speed. Random Forest receives 24 features: mean, standard deviation, min, and max for each channel across the same window.

In [ ]:
class CleanGruSpeedModel(nn.Module):
    def __init__(self, n_features: int = 6, hidden_size: int = 64, layers: int = 2, dropout: float = 0.2) -> None:
        super().__init__()
        self.gru = nn.GRU(n_features, hidden_size, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        _, hidden = self.gru(features)
        return self.head(hidden[-1]).squeeze(-1)

class CleanCnnSpeedModel(nn.Module):
    def __init__(self, n_features: int = 6, channels: tuple[int, int, int] = (32, 64, 64), dropout: float = 0.3) -> None:
        super().__init__()
        first, second, third = channels
        self.features = nn.Sequential(
            nn.Conv1d(n_features, first, kernel_size=3, padding=1), nn.BatchNorm1d(first), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(first, second, kernel_size=3, padding=1), nn.BatchNorm1d(second), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(second, third, kernel_size=3, padding=1), nn.BatchNorm1d(third), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(third, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(features.transpose(1, 2))).squeeze(-1)

assert CleanGruSpeedModel()(torch.zeros(2, CONFIG.window_size, len(VELOCITY_MODEL_FEATURE_NAMES))).shape == (2,)
assert CleanCnnSpeedModel()(torch.zeros(2, CONFIG.window_size, len(VELOCITY_MODEL_FEATURE_NAMES))).shape == (2,)

In [ ]:
# Candidate grids are deliberately small because groups, not individual windows,
# are the independent examples. Larger searches would overfit this data split.
GRU_CANDIDATES = [
    {'hidden_size': 32, 'layers': 1, 'dropout': 0.10, 'learning_rate': 1e-3},
    {'hidden_size': 64, 'layers': 2, 'dropout': 0.20, 'learning_rate': 1e-3},
    {'hidden_size': 64, 'layers': 2, 'dropout': 0.20, 'learning_rate': 5e-4},
]
CNN_CANDIDATES = [
    {'channels': (16, 32, 32), 'dropout': 0.10, 'learning_rate': 1e-3},
    {'channels': (32, 64, 64), 'dropout': 0.20, 'learning_rate': 1e-3},
    {'channels': (32, 64, 64), 'dropout': 0.30, 'learning_rate': 5e-4},
]
RF_CANDIDATES = [
    {'n_estimators': 100, 'max_depth': 12, 'min_samples_leaf': 1},
    {'n_estimators': 100, 'max_depth': 16, 'min_samples_leaf': 1},
    {'n_estimators': 100, 'max_depth': 16, 'min_samples_leaf': 3},
    {'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 5},
]

def scale_windows(scaler: StandardScaler, windows: np.ndarray) -> np.ndarray:
    return scaler.transform(windows.reshape(-1, windows.shape[-1])).reshape(windows.shape).astype(np.float32)

def window_stats(windows: np.ndarray) -> np.ndarray:
    return np.concatenate([windows.mean(axis=1), windows.std(axis=1), windows.min(axis=1), windows.max(axis=1)], axis=1)

def macro_journey_mae(actual: np.ndarray, predicted: np.ndarray, journey_ids: np.ndarray) -> float:
    return float(np.mean([mean_absolute_error(actual[journey_ids == journey], predicted[journey_ids == journey]) for journey in np.unique(journey_ids)]))

def make_neural_model(family: str, parameters: dict) -> nn.Module:
    if family == 'gru':
        return CleanGruSpeedModel(hidden_size=parameters['hidden_size'], layers=parameters['layers'], dropout=parameters['dropout'])
    if family == 'cnn':
        return CleanCnnSpeedModel(channels=parameters['channels'], dropout=parameters['dropout'])
    raise ValueError(f'Unknown neural family: {family}')

def make_loader(features: np.ndarray, targets: np.ndarray, shuffle: bool) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(targets))
    return DataLoader(dataset, batch_size=CONFIG.batch_size, shuffle=shuffle, num_workers=0, pin_memory=DEVICE.type == 'cuda')

def scaled_neural_predictions(model: nn.Module, features: np.ndarray) -> np.ndarray:
    model.eval()
    values = []
    with torch.inference_mode():
        for start in range(0, len(features), CONFIG.batch_size):
            batch = torch.from_numpy(features[start:start + CONFIG.batch_size]).to(DEVICE)
            values.append(model(batch).cpu().numpy())
    return np.concatenate(values)

def tune_neural_family(family: str, candidates: list[dict]) -> pd.DataFrame:
    rows = []
    for candidate_id, parameters in enumerate(candidates):
        fold_scores, fold_epochs = [], []
        for fold_id, (fold_train, fold_validation) in enumerate(cv_folds):
            seed_everything(CONFIG.seed + 100 * candidate_id + fold_id)
            x_scaler = StandardScaler().fit(X[fold_train].reshape(-1, X.shape[-1]))
            y_scaler = StandardScaler().fit(y_mps[fold_train].reshape(-1, 1))
            x_train, x_validation = scale_windows(x_scaler, X[fold_train]), scale_windows(x_scaler, X[fold_validation])
            y_train = y_scaler.transform(y_mps[fold_train].reshape(-1, 1)).astype(np.float32).ravel()
            model = make_neural_model(family, parameters).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=parameters['learning_rate'], weight_decay=CONFIG.weight_decay)
            loss_function = nn.MSELoss()
            best_state, best_score, stale, best_epoch = None, float('inf'), 0, 0
            for epoch in range(1, CONFIG.tuning_max_epochs + 1):
                model.train()
                for features, target in make_loader(x_train, y_train, True):
                    optimizer.zero_grad(set_to_none=True)
                    loss = loss_function(model(features.to(DEVICE)), target.to(DEVICE))
                    loss.backward()
                    optimizer.step()
                predicted = y_scaler.inverse_transform(scaled_neural_predictions(model, x_validation).reshape(-1, 1)).ravel()
                score = macro_journey_mae(y_mps[fold_validation], predicted, groups[fold_validation].astype(str))
                if score < best_score - 1e-6:
                    best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
                    best_score, best_epoch, stale = score, epoch, 0
                else:
                    stale += 1
                    if stale >= CONFIG.tuning_early_stopping_patience:
                        break
            assert best_state is not None
            fold_scores.append(best_score)
            fold_epochs.append(best_epoch)
            print(f'{family} candidate {candidate_id}, fold {fold_id}: macro MAE={best_score:.3f} m/s at epoch {best_epoch}')
        rows.append({'model': family, 'candidate_id': candidate_id, 'parameters': parameters, 'cv_macro_mae_mps': float(np.mean(fold_scores)), 'cv_macro_mae_std_mps': float(np.std(fold_scores)), 'fold_best_epochs': fold_epochs})
    return pd.DataFrame(rows).sort_values('cv_macro_mae_mps').reset_index(drop=True)

def tune_random_forest() -> pd.DataFrame:
    rows = []
    for candidate_id, parameters in enumerate(RF_CANDIDATES):
        fold_scores = []
        for fold_id, (fold_train, fold_validation) in enumerate(cv_folds):
            x_scaler = StandardScaler().fit(X[fold_train].reshape(-1, X.shape[-1]))
            forest = RandomForestRegressor(**parameters, n_jobs=4, random_state=CONFIG.seed + candidate_id)
            forest.fit(window_stats(scale_windows(x_scaler, X[fold_train])), y_mps[fold_train])
            predicted = forest.predict(window_stats(scale_windows(x_scaler, X[fold_validation])))
            score = macro_journey_mae(y_mps[fold_validation], predicted, groups[fold_validation].astype(str))
            fold_scores.append(score)
            print(f'rf candidate {candidate_id}, fold {fold_id}: macro MAE={score:.3f} m/s')
        rows.append({'model': 'random_forest', 'candidate_id': candidate_id, 'parameters': parameters, 'cv_macro_mae_mps': float(np.mean(fold_scores)), 'cv_macro_mae_std_mps': float(np.std(fold_scores)), 'fold_best_epochs': []})
    return pd.DataFrame(rows).sort_values('cv_macro_mae_mps').reset_index(drop=True)

def regression_metrics(actual: np.ndarray, predicted: np.ndarray, journey_ids: np.ndarray) -> dict[str, float]:
    return {'mae_mps': float(mean_absolute_error(actual, predicted)), 'rmse_mps': float(mean_squared_error(actual, predicted) ** 0.5), 'mae_kmh': float(mean_absolute_error(actual, predicted) * 3.6), 'rmse_kmh': float(mean_squared_error(actual, predicted) ** 0.5 * 3.6), 'macro_journey_mae_mps': macro_journey_mae(actual, predicted, journey_ids.astype(str)), 'r2': float(r2_score(actual, predicted))}

def evaluate_model(model: nn.Module, data_loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    predictions, actual = [], []
    with torch.no_grad():
        for features, target in data_loader:
            predictions.append(model(features.to(DEVICE)).cpu().numpy())
            actual.append(target.numpy())
    predicted_mps = target_scaler.inverse_transform(np.concatenate(predictions).reshape(-1, 1)).ravel()
    actual_mps = target_scaler.inverse_transform(np.concatenate(actual).reshape(-1, 1)).ravel()
    return actual_mps, predicted_mps

def train_model(name: str, model: nn.Module) -> tuple[nn.Module, pd.DataFrame]:
    seed_everything(CONFIG.seed)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
    loss_function = nn.MSELoss()
    best_state, best_validation, stale_epochs, history = None, float('inf'), 0, []
    start = time.perf_counter()
    for epoch in range(1, CONFIG.max_epochs + 1):
        model.train()
        train_losses = []
        for features, target in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(features.to(DEVICE)), target.to(DEVICE))
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))
        actual, predicted = evaluate_model(model, validation_loader)
        validation_mae = mean_absolute_error(actual, predicted)
        history.append({'epoch': epoch, 'train_mse_scaled': float(np.mean(train_losses)), 'validation_mae_mps': validation_mae, 'elapsed_s': time.perf_counter() - start})
        print(f'{name} epoch {epoch:02d}: train MSE={history[-1]["train_mse_scaled"]:.4f}; validation MAE={validation_mae:.3f} m/s')
        if validation_mae < best_validation:
            best_validation, stale_epochs = validation_mae, 0
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        else:
            stale_epochs += 1
            if stale_epochs >= CONFIG.early_stopping_patience:
                print(f'{name}: early stop after {epoch} epochs.')
                break
    assert best_state is not None
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
# Model selection: CV sees development journeys only; the test set remains
# untouched until all three best candidates are refit below.
rf_cv = tune_random_forest()
gru_cv = tune_neural_family('gru', GRU_CANDIDATES)
cnn_cv = tune_neural_family('cnn', CNN_CANDIDATES)
selection_results = pd.concat([rf_cv, gru_cv, cnn_cv], ignore_index=True).sort_values('cv_macro_mae_mps').reset_index(drop=True)
print(selection_results[['model', 'candidate_id', 'parameters', 'cv_macro_mae_mps', 'cv_macro_mae_std_mps', 'fold_best_epochs']].to_string(index=False))

def refit_neural(family: str, parameters: dict, epochs: int) -> tuple[nn.Module, pd.DataFrame]:
    seed_everything(CONFIG.seed)
    model = make_neural_model(family, parameters).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=parameters['learning_rate'], weight_decay=CONFIG.weight_decay)
    loss_function = nn.MSELoss()
    history, start = [], time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for features, target in make_loader(X_development_scaled, y_development_scaled, True):
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(features.to(DEVICE)), target.to(DEVICE))
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        history.append({'epoch': epoch, 'train_mse_scaled': float(np.mean(losses)), 'elapsed_s': time.perf_counter() - start})
    return model, pd.DataFrame(history)

# Final scalers are fit only on development journeys. They are the scalers
# that ship with the selected models; frozen test data never influences them.
feature_scaler = StandardScaler().fit(X[development_index].reshape(-1, X.shape[-1]))
target_scaler = StandardScaler().fit(y_mps[development_index].reshape(-1, 1))
X_development_scaled = scale_windows(feature_scaler, X[development_index])
y_development_scaled = target_scaler.transform(y_mps[development_index].reshape(-1, 1)).astype(np.float32).ravel()
X_test_scaled = scale_windows(feature_scaler, X[test_index])

selected = {family: selection_results.loc[selection_results['model'] == family].iloc[0].to_dict() for family in ('gru', 'cnn', 'random_forest')}
trained, histories, predictions, metrics = {}, {}, {}, []
for family in ('gru', 'cnn'):
    selected_epochs = int(np.clip(round(np.median(selected[family]['fold_best_epochs'])), 1, CONFIG.final_max_epochs))
    trained[family], histories[family] = refit_neural(family, selected[family]['parameters'], selected_epochs)
    predicted = target_scaler.inverse_transform(scaled_neural_predictions(trained[family], X_test_scaled).reshape(-1, 1)).ravel()
    predictions[family] = predicted
    metrics.append({'model': family, **regression_metrics(y_mps[test_index], predicted, groups[test_index]), 'cv_macro_mae_mps': selected[family]['cv_macro_mae_mps'], 'epochs': selected_epochs, 'training_s': float(histories[family]['elapsed_s'].iloc[-1])})

rf_parameters = {**selected['random_forest']['parameters'], 'n_estimators': 300}
trained['random_forest'] = RandomForestRegressor(**rf_parameters, n_jobs=4, random_state=CONFIG.seed)
rf_start = time.perf_counter()
trained['random_forest'].fit(window_stats(X_development_scaled), y_mps[development_index])
rf_training_s = time.perf_counter() - rf_start
predictions['random_forest'] = trained['random_forest'].predict(window_stats(X_test_scaled))
metrics.append({'model': 'random_forest', **regression_metrics(y_mps[test_index], predictions['random_forest'], groups[test_index]), 'cv_macro_mae_mps': selected['random_forest']['cv_macro_mae_mps'], 'epochs': None, 'training_s': rf_training_s})

comparison = pd.DataFrame(metrics).sort_values('macro_journey_mae_mps').reset_index(drop=True)
print(comparison.to_string(index=False))

In [ ]:
# Measure end-to-end model inference after a clean window is available:
# feature scaling, model execution, and target inverse-scaling. This is not
# a measurement of the deterministic preprocessing latency.
def synchronize_device() -> None:
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

def neural_predict_one(model: nn.Module, raw_window: np.ndarray) -> float:
    scaled = scale_windows(feature_scaler, raw_window[None, ...])
    with torch.inference_mode():
        value = float(model(torch.from_numpy(scaled).to(DEVICE)).cpu().item())
    return float(target_scaler.inverse_transform([[value]])[0, 0])

def forest_predict_one(model: RandomForestRegressor, raw_window: np.ndarray) -> float:
    scaled = scale_windows(feature_scaler, raw_window[None, ...])
    return float(model.predict(window_stats(scaled))[0])

def measure_single_window_latency(predict_one) -> float:
    sample = X[test_index[0]]
    for _ in range(CONFIG.inference_warmup_calls):
        predict_one(sample)
    synchronize_device()
    start = time.perf_counter()
    for _ in range(CONFIG.inference_measurement_calls):
        predict_one(sample)
    synchronize_device()
    return (time.perf_counter() - start) * 1e3 / CONFIG.inference_measurement_calls

inference_rows = []
for family in ('gru', 'cnn'):
    trained[family].eval()
    inference_rows.append({'model': family, 'device': str(DEVICE), 'single_window_end_to_end_ms': measure_single_window_latency(lambda window, model=trained[family]: neural_predict_one(model, window))})
inference_rows.append({'model': 'random_forest', 'device': 'cpu', 'single_window_end_to_end_ms': measure_single_window_latency(lambda window: forest_predict_one(trained['random_forest'], window))})
inference_comparison = pd.DataFrame(inference_rows)
comparison = comparison.merge(inference_comparison, on='model', how='left')
print(inference_comparison.to_string(index=False))

# Save a complete experiment bundle: selected weights, scalers, CV evidence,
# held-out metrics, predictions, and the inference-latency methodology.
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({'feature_scaler': feature_scaler, 'target_scaler': target_scaler}, ARTIFACT_DIR / 'scalers.joblib')
for family in ('gru', 'cnn'):
    torch.save({'model_name': family, 'state_dict': trained[family].state_dict(), 'parameters': selected[family]['parameters'], 'window_size': CONFIG.window_size, 'n_features': len(VELOCITY_MODEL_FEATURE_NAMES), 'feature_names': list(VELOCITY_MODEL_FEATURE_NAMES), 'target_unit': 'm/s'}, ARTIFACT_DIR / f'{family}_clean_velocity.pt')
    histories[family].to_csv(ARTIFACT_DIR / f'{family}_final_history.csv', index=False)
joblib.dump(trained['random_forest'], ARTIFACT_DIR / 'random_forest_clean_velocity.joblib')
selection_results.to_csv(ARTIFACT_DIR / 'cv_selection.csv', index=False)
comparison.to_csv(ARTIFACT_DIR / 'comparison.csv', index=False)
inference_comparison.to_csv(ARTIFACT_DIR / 'inference_latency.csv', index=False)
pd.DataFrame({'sequence_id': groups[test_index].astype(str), 'actual_mps': y_mps[test_index], **{f'{name}_prediction_mps': values for name, values in predictions.items()}}).to_csv(ARTIFACT_DIR / 'test_predictions.csv', index=False)
audit_frame.to_csv(ARTIFACT_DIR / 'sequence_audit.csv', index=False)
skipped_frame.to_csv(ARTIFACT_DIR / 'skipped_sequences.csv', index=False)
manifest = {'config': asdict(CONFIG), 'feature_names': list(VELOCITY_MODEL_FEATURE_NAMES), 'feature_unit_order': ['m/s^2', 'm/s^2', 'm/s^2', 'rad/s', 'rad/s', 'rad/s'], 'target_unit': 'm/s', 'development_journeys': sorted(map(str, set(groups[development_index]))), 'test_journeys': sorted(map(str, set(groups[test_index]))), 'selected_parameters': {**{name: selected[name]['parameters'] for name in ('gru', 'cnn')}, 'random_forest': rf_parameters}, 'cv_selection_metric': 'macro journey MAE in m/s', 'inference_latency_definition': 'single clean 50x6 window, feature scaling + model + target inverse-scaling; deterministic preprocessing excluded', 'notes': ['Models were retrained from scratch on deterministic-pipeline output.', 'Rolling calibration evidence uses phone GPS speed, not the CAN-bus target.', 'The offline evidence heuristic is causal but must be replaced by validated production evidence rules before deployment.']}
(ARTIFACT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

plt.figure(figsize=(8, 4))
plt.bar(comparison['model'], comparison['macro_journey_mae_mps'])
plt.ylabel('frozen-test macro journey MAE (m/s)'); plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'test_macro_mae.png', dpi=150)
# Close the saved figure so command-line execution never waits for a GUI.
plt.close()
print(f'Saved reproducible comparison bundle to {ARTIFACT_DIR}')

## Interpreting the result

Choose a model first by the three-fold development CV macro journey MAE, then report its frozen unseen-journey test metrics once. Do not select a winner using its test result. The Random Forest is an intentionally simpler 24-statistic baseline; if it wins, that is a useful finding rather than a failure.

The latency number measures prediction after a clean window exists. End-to-end navigation latency also includes deterministic preprocessing, window accumulation, uncertainty prediction, and EKF work; benchmark those separately. Confirm the saved manifest, scalers, feature order, units, and window shape accompany any selected checkpoint before the velocity adapter uses it.

After this comparison, the next engineering step is to expose the winner through adapters/velocity_predictor.py together with its scalers and to feed its prediction into the separately trained uncertainty engine. Do not train the uncertainty engine inside this notebook.